In [1]:
import pandas as pd
import numpy as np
import shutil

In [ ]:
excel_path = r".Data/final_scores.xlsx"
output_path = r"./Data/consolidated_results.xlsx"

# List of sheets to evaluate (skipping overview sheets like 'Whole' if present)
# sheet_mapping = {
#     "DP": "Data Provenance",
#     "PR": "Process Replication",
#     "AE": "Authority & Expertise",
#     "DV": "Data Volume",
#     "AF": "Anti-Forensics",
#     "CL": "Cross-Linking",
#     "DA": "Data Availability"
# }

sheet_mapping = {
    "CoC": "Chain of Custody",
    "Au": "Authenticity",
    "In": "Integrity",
    "Re": "Reliability",
    "La": "Lawful Acquisition",
    "Fi": "Fact vs Inference",
    "Cq": "Competence and Qualification",
    "Rr": "Repeatability and reproducibility",
    "Pc": "Privacy compliance"
}

# Configuration Settings
annotator_cols = ["annotator1", "annotator2", "annotator3"]
scale = ["0", "1", "2"]

In [7]:
# Load the Whole sheet
whole_df = pd.read_excel(excel_path, sheet_name="Whole")
whole_df["Publication"] = whole_df["Publication"].astype(str).str.strip()

print(f"{'Publication':<40} | {'Column':<35} | {'Old Value':<10} | {'New Value':<10}")
print("-" * 105)

for sheet_name, column_name in sheet_mapping.items():
    sheet = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])

    sheet.columns = pd.MultiIndex.from_tuples(
        [(str(c[0]).strip(), str(c[1]).strip()) for c in sheet.columns]
    )

    pub_col_tuple = [col for col in sheet.columns if col[0].lower() == "publication"][0]
    publications = sheet[pub_col_tuple].astype(str).str.strip()

    r2_annotators = [
        col for col in sheet["Round 2"].columns 
        if "annotator" in col.lower()
    ]

    round_2_data = sheet["Round 2"][r2_annotators].apply(pd.to_numeric, errors="coerce")
    medians = round_2_data.median(axis=1, skipna=True)

    median_map = pd.Series(medians.values, index=publications).dropna()

    target_col = next(
        (c for c in whole_df.columns if c.strip().lower() == column_name.lower()), 
        None
    )
    if target_col is None:
        print(f"Warning: Column '{column_name}' not found in 'Whole'. Skipping {sheet_name}.")
        continue

    new_values = whole_df["Publication"].map(median_map)
    current_numeric = pd.to_numeric(whole_df[target_col], errors="coerce")
    
    # Identify rows that actually need an update
    mismatch_mask = new_values.notna() & ~np.isclose(current_numeric, new_values, equal_nan=True)

    # Print changes before applying them
    if mismatch_mask.any():
        changed_rows = whole_df.loc[mismatch_mask, ["Publication", target_col]].copy()
        changed_rows["New_Value"] = new_values[mismatch_mask]

        for _, row in changed_rows.iterrows():
            pub = row["Publication"]
            old_val = row[target_col]
            new_val = row["New_Value"]
            print(f"{pub:<40} | {target_col:<35} | {str(old_val):<10} | {str(new_val):<10}")

        # Update whole_df
        whole_df.loc[mismatch_mask, target_col] = new_values[mismatch_mask]

# Preserve all existing sheets by copying original file and replacing 'Whole'
shutil.copyfile(excel_path, output_path)

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    whole_df.to_excel(writer, sheet_name="Whole", index=False)

print("-" * 105)
print("Consolidation complete.")

Publication                              | Column                              | Old Value  | New Value 
---------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------
Consolidation complete.


In [8]:
whole_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Publication              7 non-null      str   
 1   Unnamed: 1               96 non-null     str   
 2   Legislative Requirement  28 non-null     object
 3   Unnamed: 3               28 non-null     object
 4   Unnamed: 4               28 non-null     object
 5   Unnamed: 5               28 non-null     object
 6   Unnamed: 6               28 non-null     object
 7   Unnamed: 7               28 non-null     object
 8   Unnamed: 8               28 non-null     object
 9   Unnamed: 9               28 non-null     object
 10  Unnamed: 10              28 non-null     object
dtypes: object(9), str(2)
memory usage: 8.4+ KB


In [9]:
def get_row_median(
    row: pd.Series,
    r2_cols: list,
    r1_cols: list,
) -> float:
    """Compute the consensus score for a single row across rating rounds.

    Evaluates scores by prioritizing Round 2 ratings whenever at least
    one valid numeric score exists. If Round 2 contains no valid entries,
    falls back to Round 1. Resolves even-length score ties by selecting
    the lower middle rank using floor division index (n - 1) // 2.

    Args:
        row: Pandas Series representing an observation row with ratings.
        r2_cols: Sequence of column identifiers corresponding to Round 2
            annotators.
        r1_cols: Sequence of column identifiers corresponding to Round 1
            annotators.

    Returns:
        The consolidated median score as a float, or np.nan if neither
        round contains valid numeric ratings.

    Example:
        >>> import pandas as pd
        >>> import numpy as np
        >>> sample_row = pd.Series({"r2_a": 2, "r2_b": 3, "r1_a": 1})
        >>> get_row_median(sample_row, ["r2_a", "r2_b"], ["r1_a"])
        2.0
    """
    def extract_scores(cols: list) -> list[float]:
        """Extract and coerce non-null numeric ratings from target columns.

        Args:
            cols: List of column identifiers to inspect in the parent row.

        Returns:
            List of valid float scores extracted from the row.
        """
        scores = []
        for col in cols:
            val = pd.to_numeric(row[col], errors="coerce")
            if pd.notna(val):
                scores.append(val)
        return scores

    active_scores = extract_scores(r2_cols)

    # Fall back to Round 1 when Round 2 has no ratings
    if not active_scores:
        active_scores = extract_scores(r1_cols)

    if not active_scores:
        return np.nan

    active_scores.sort()
    n = len(active_scores)
    # Select lower middle index to resolve even-length ties deterministically
    return active_scores[(n - 1) // 2]


# Load Whole Sheet
whole_preview = pd.read_excel(excel_path, sheet_name="Whole", nrows=2, header=None)
is_multi = False
if len(whole_preview) >= 2:
    row0 = " ".join(whole_preview.iloc[0].dropna().astype(str).str.lower())
    row1 = " ".join(whole_preview.iloc[1].dropna().astype(str).str.lower())
    if "publication" in row0 and ("author" in row1 or "category" in row1):
        is_multi = True

whole_df = pd.read_excel(excel_path, sheet_name="Whole", header=[0, 1] if is_multi else 0)

if is_multi:
    whole_df.columns = pd.MultiIndex.from_tuples(
        [(str(c[0]).strip(), str(c[1]).strip()) for c in whole_df.columns]
    )
    whole_author_col = next(
        (c for c in whole_df.columns if c[0].lower() == "publication" and c[1].lower() == "author"),
        next((c for c in whole_df.columns if "author" in str(c[1]).lower()), None)
    )
    whole_cat_col = next(
        (c for c in whole_df.columns if c[0].lower() == "publication" and c[1].lower() == "category"),
        None
    )
else:
    whole_author_col = next(
        (c for c in whole_df.columns if str(c).strip().lower() in ["author", "publication"]),
        whole_df.columns[0]
    )
    whole_cat_col = next((c for c in whole_df.columns if str(c).strip().lower() == "category"), None)

# Clean and normalize strings
whole_df[whole_author_col] = whole_df[whole_author_col].astype(str).str.strip()
if whole_cat_col:
    # Forward-fill merged category cells so each author row retains its category
    whole_df[whole_cat_col] = whole_df[whole_cat_col].replace(r"^\s*$", np.nan, regex=True).ffill()

print(f"{'Publication':<40} | {'Column':<35} | {'Old Value':<10} | {'New Value':<10}")
print("-" * 105)

# Keep track of target columns that are updated
updated_target_cols = []

# Iterate Sheets & Consolidate Scores
for sheet_name, column_name in sheet_mapping.items():
    sheet = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])
    sheet.columns = pd.MultiIndex.from_tuples(
        [(str(c[0]).strip(), str(c[1]).strip()) for c in sheet.columns]
    )

    # Locate Author column
    sheet_author_col = next(
        (c for c in sheet.columns if c[0].lower() == "publication" and c[1].lower() == "author"),
        next((c for c in sheet.columns if c[0].lower() == "publication" and "category" not in c[1].lower()), None)
    )
    if not sheet_author_col:
        sheet_author_col = next((c for c in sheet.columns if "author" in str(c[1]).lower()), None)

    if not sheet_author_col:
        print(f"Warning: Could not identify Author column in '{sheet_name}'. Skipping.")
        continue

    # Locate Annotator columns for Round 2 and Round 1
    r2_annotators = [
        c for c in sheet.columns 
        if ("round 2" in str(c[0]).lower() or str(c[0]).lower() == "round 2") and 
           ("annotator" in str(c[1]).lower() or str(c[1]).lower() in ["annotator1", "annotator2", "annotator3"])
    ]
    r1_annotators = [
        c for c in sheet.columns 
        if ("round 1" in str(c[0]).lower() or str(c[0]).lower() == "round 1") and 
           ("annotator" in str(c[1]).lower() or str(c[1]).lower() in ["annotator1", "annotator2", "annotator3"])
    ]

    # Calculate median scores for this sheet
    median_map = {}
    for _, row in sheet.iterrows():
        pub = str(row[sheet_author_col]).strip()
        if not pub or pub.lower() in ["nan", "none"]:
            continue

        score = get_row_median(row, r2_annotators, r1_annotators)
        if pd.notna(score):
            median_map[pub] = score

    # Locate target column in whole_df
    if is_multi:
        target_col = next(
            (c for c in whole_df.columns if str(c[0]).strip().lower() == column_name.strip().lower() or 
                                           str(c[1]).strip().lower() == column_name.strip().lower()),
            None
        )
    else:
        target_col = next(
            (c for c in whole_df.columns if str(c).strip().lower() == column_name.strip().lower()),
            None
        )

    if target_col is None:
        print(f"Warning: Column '{column_name}' not found in 'Whole'. Skipping {sheet_name}.")
        continue

    updated_target_cols.append(target_col)

    # Map scores to whole_df
    new_values = whole_df[whole_author_col].map(median_map)
    current_numeric = pd.to_numeric(whole_df[target_col], errors="coerce")

    # Only update when a valid new score was calculated and differs from current
    mismatch_mask = new_values.notna() & ~np.isclose(current_numeric, new_values, equal_nan=True)

    if mismatch_mask.any():
        changed_rows = whole_df.loc[mismatch_mask, [whole_author_col, target_col]].copy()
        changed_rows["New_Value"] = new_values[mismatch_mask]

        col_display = target_col if isinstance(target_col, str) else " - ".join(target_col)
        for _, row in changed_rows.iterrows():
            pub = row[whole_author_col]
            old_val = row[target_col]
            new_val = row["New_Value"]
            print(f"{pub:<40} | {col_display:<35} | {str(old_val):<10} | {str(new_val):<10}")

        whole_df.loc[mismatch_mask, target_col] = new_values[mismatch_mask]

# Filter Final Dataframe: Keep Any Row With >= 1 Valid Score
if updated_target_cols:
    # Check if a row has at least one valid score across any evaluated metric column
    has_valid_entry = (
        whole_df[updated_target_cols]
        .apply(pd.to_numeric, errors="coerce")
        .notna()
        .any(axis=1)
    )
    
    total_rows_before = len(whole_df)
    whole_df = whole_df.loc[has_valid_entry].reset_index(drop=True)
    dropped_count = total_rows_before - len(whole_df)
    
    if dropped_count > 0:
        print(f"\nDropped {dropped_count} row(s) that had zero entries across all categories/metrics.")

# Save Finalized Dataframe
shutil.copyfile(excel_path, output_path)

# Set the publication identification columns as index so they format cleanly on export
pub_cols = [
    col for col in whole_df.columns 
    if isinstance(col, tuple) and str(col[0]).strip().lower() == "publication"
]

if pub_cols:
    export_df = whole_df.set_index(pub_cols)
    write_index = True
else:
    export_df = whole_df
    write_index = False if not isinstance(whole_df.columns, pd.MultiIndex) else True

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    export_df.to_excel(writer, sheet_name="Whole", index=write_index)

print("-" * 105)
print("Consolidation complete.")


Publication                              | Column                              | Old Value  | New Value 
---------------------------------------------------------------------------------------------------------
Li_2023                                  | Legislative Requirement - Chain of Custody | 2.0        |     1.0
Name: 6, dtype: object
Sathyaprakasan_2021                      | Legislative Requirement - Chain of Custody | 0.0        |     2.0
Name: 12, dtype: object
Wagner_2023                              | Legislative Requirement - Chain of Custody | 2.0        |     1.0
Name: 13, dtype: object
Hao_2018                                 | Legislative Requirement - Chain of Custody | 2.0        |     0.0
Name: 15, dtype: object
Frowis_2020                              | Legislative Requirement - Chain of Custody | 2.0        |     1.0
Name: 16, dtype: object
Ellul_2020                               | Legislative Requirement - Chain of Custody | 2.0        |     0.0
Name: 17, dtype:

In [5]:
whole_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 11 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   (Publication, Category)                                       95 non-null     str    
 1   (Publication, Author)                                         95 non-null     str    
 2   (Legislative Requirement, Chain of Custody)                   27 non-null     float64
 3   (Legislative Requirement, Authenticity)                       27 non-null     float64
 4   (Legislative Requirement, Integrity)                          27 non-null     float64
 5   (Legislative Requirement, Reliability)                        27 non-null     float64
 6   (Legislative Requirement, Lawful Acquisition)                 27 non-null     float64
 7   (Legislative Requirement, Fact vs Inference)                  27 non-null     float6